# RAASTA Pass 5: big balanced retrain (all 11 classes)

**Goal:** every class as reliable as pothole on the phone. More data per class, balanced so no class drowns another, and a real test set (150+ images per class) so the scores mean something.

**Two sittings are fine.** Build tonight on a normal CPU runtime (cells 1-9; cell 9 saves the dataset to Drive). Next day: switch to T4 GPU, run cell 1, cell 2, Restore, then cells 10 and 11.

Everything downloads to the Colab disk (`/content`), which is fast. Only the finished dataset zip, the weights and the results go to Drive. Pass 4 died reading 21k photos from Drive; this notebook never does that.

| Step | Cell | Time |
|------|------|------|
| 1 | Setup | 2 min |
| 2 | Helpers | instant |
| 3 | Road damage: RDD2022 India + Japan | 10-15 min |
| 4 | Roboflow sets (paste API key) | 15-25 min |
| 5 | Pothole zip + African wildlife (buffalo) | 3 min |
| 6 | COCO street people + dog, cat, horse, cow | 10-15 min |
| 7 | Open Images goat, donkey, horse, cow | 10-15 min |
| 8 | Balance + write dataset (prints counts per class) | 10 min |
| 9 | Save dataset zip to Drive (for resume) | 5 min |
| 10 | Old model score on the new test set | 3 min |
| 11 | **Train (overnight)** | about 3-4 hours |
| 12 | Resume (only if Colab disconnected) | |
| 13 | Old vs new, per class | 3 min |
| 14 | Export phone file | 5 min |

If a single download fails, the cell prints `skip` and continues. Only stop if cell 8 prints a class under 300 images.

**Send me in the morning:** the tables printed by cells 8 and 13.

In [ ]:
#@title 1) Setup
from google.colab import drive
drive.mount('/content/drive')

import subprocess, sys
from pathlib import Path
import torch

ROOT = Path('/content/drive/MyDrive/raasta')
WEIGHTS = ROOT / 'weights'
EXPORTS = ROOT / 'exports'
RAW_DRIVE = ROOT / 'raw'
for p in (WEIGHTS, EXPORTS, RAW_DRIVE):
    p.mkdir(parents=True, exist_ok=True)

WORK = Path('/content/p5')
DL = WORK / 'downloads'
DL.mkdir(parents=True, exist_ok=True)
MIX = Path('/content/pass5_mix')
DATASET_ZIP = RAW_DRIVE / 'pass5_dataset.zip'
RUN = WEIGHTS / 'm2_m5_pass5'
RESULTS = RUN / 'results'
RESULTS.mkdir(parents=True, exist_ok=True)

OLD_CANDIDATES = [
    WEIGHTS / 'm2_m5_balanced_v1/finetune/weights/best.pt',
    WEIGHTS / 'm2_m5_night_public_v1/finetune/weights/best.pt',
    WEIGHTS / 'm2_m5_people_animals_v1/finetune/weights/best.pt',
]
OLD_BEST = next((p for p in OLD_CANDIDATES if p.exists()), None)
if OLD_BEST is None:
    found = sorted(
        (p for p in WEIGHTS.rglob('best.pt') if 'm2_m5_pass5' not in str(p)),
        key=lambda p: p.stat().st_mtime, reverse=True,
    )
    OLD_BEST = found[0] if found else None

subprocess.check_call([sys.executable, '-m', 'pip', '-q', 'install', 'ultralytics', 'roboflow', 'pyyaml'])

print('GPU', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')
print('Start model', OLD_BEST)
print('Dataset zip on Drive', DATASET_ZIP.exists())
print('Pass 5 last.pt', (RUN / 'finetune/weights/last.pt').exists())
if not torch.cuda.is_available():
    print('CPU runtime: fine for building the dataset (cells 2-9). Switch to T4 GPU before cell 10.')
assert OLD_BEST is not None, 'No previous best.pt found under MyDrive/raasta/weights'

## 2) Helpers

Every source goes into one pool. Class names from each dataset are matched by keyword (for example `Cattle` and `bull` become `cow`, `mule` becomes `donkey`). `rider` is not mapped to person on purpose, so motorbike riders do not trigger "person" alerts.

In [ ]:
#@title 2) Helpers
import os, re, csv, json, random, shutil, zipfile, time
import xml.etree.ElementTree as ET
import urllib.request
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed
import yaml
from PIL import Image

CLASS_NAMES = [
    'pothole', 'crack', 'speed_bump', 'person', 'cow', 'buffalo',
    'dog', 'cat', 'horse', 'donkey', 'goat',
]
CID = {n: i for i, n in enumerate(CLASS_NAMES)}
IMG_EXT = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

KEYWORDS = [
    ('pothole', ['pothole', 'potholes', 'pot hole', 'd40']),
    ('crack', ['crack', 'cracks', 'd00', 'd10', 'd20', 'alligator crack',
               'longitudinal crack', 'transverse crack']),
    ('speed_bump', ['speed bump', 'speed bumps', 'speedbump', 'speed breaker',
                    'speed hump', 'hump', 'bump', 'unmarked bump', 'marked bump']),
    ('buffalo', ['buffalo', 'water buffalo', 'bufflao', 'buffallo', 'kerbau', 'murrah']),
    ('cow', ['cow', 'cows', 'cattle', 'bull', 'ox', 'calf']),
    ('donkey', ['donkey', 'donkeys', 'mule']),
    ('horse', ['horse', 'horses', 'pony']),
    ('goat', ['goat', 'goats']),
    ('dog', ['dog', 'dogs', 'puppy']),
    ('cat', ['cat', 'cats', 'kitten']),
    ('person', ['person', 'persons', 'people', 'pedestrian', 'pedestrians',
                'human', 'man', 'woman', 'child', 'boy', 'girl']),
]

def canon(raw):
    s = ' '.join(re.sub(r'[_\-]+', ' ', str(raw).lower()).split())
    for name, kws in KEYWORDS:
        for kw in kws:
            if re.search(r'(?<![a-z])' + re.escape(kw) + r'(?![a-z])', s):
                return name
    return None

POOL = []          # (image path, [yolo lines], source)
BG = []            # (image path, source) road photos with no hazard
SRC_STATS = {}

def download(url, dest, min_bytes=1_000_000):
    dest = Path(dest)
    dest.parent.mkdir(parents=True, exist_ok=True)
    if dest.exists() and dest.stat().st_size >= min_bytes:
        print('have', dest.name, f'{dest.stat().st_size/1e6:.0f} MB')
        return dest
    if dest.exists():
        dest.unlink()
    print('downloading', dest.name)
    cmd = ['curl', '-L', '--fail', '--retry', '8', '--retry-all-errors',
           '--retry-delay', '3', '-A', 'Mozilla/5.0', '-o', str(dest), url]
    try:
        subprocess.check_call(cmd)
    except subprocess.CalledProcessError:
        subprocess.check_call(['wget', '-c', '--tries=8', '-U', 'Mozilla/5.0', '-O', str(dest), url])
    if dest.stat().st_size < min_bytes:
        raise RuntimeError(f'download too small: {dest}')
    return dest

def unzip(zip_path, out_dir):
    out_dir = Path(out_dir)
    marker = out_dir / '_OK'
    if marker.exists():
        return out_dir
    out_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(out_dir)
    marker.write_text('ok')
    return out_dir

def read_names(root):
    for y in Path(root).rglob('data.yaml'):
        names = (yaml.safe_load(y.read_text()) or {}).get('names')
        if isinstance(names, dict):
            return {int(k): str(v) for k, v in names.items()}
        if isinstance(names, list):
            return {i: str(n) for i, n in enumerate(names)}
    return None

def yolo_pairs(root):
    out = []
    for img in Path(root).rglob('*'):
        if img.suffix.lower() not in IMG_EXT:
            continue
        s = str(img)
        i = s.rfind('/images')
        if i < 0:
            continue
        lbl = Path(s[:i] + '/labels' + s[i + 7:]).with_suffix('.txt')
        if lbl.exists():
            out.append((img, lbl))
    return out

def yolo_line(cls_id, cx, cy, w, h):
    cx, cy = min(max(cx, 0.0), 1.0), min(max(cy, 0.0), 1.0)
    w, h = min(max(w, 0.0), 1.0), min(max(h, 0.0), 1.0)
    if w < 0.003 or h < 0.003:
        return None
    return f'{cls_id} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}'

def remap_file(lbl, names, force=None, min_h=0.0):
    out = []
    for line in lbl.read_text(encoding='utf-8', errors='ignore').splitlines():
        parts = line.split()
        if len(parts) < 5:
            continue
        try:
            raw_id = int(float(parts[0]))
            vals = [float(v) for v in parts[1:]]
        except ValueError:
            continue
        name = force or canon(names.get(raw_id, raw_id) if names else raw_id)
        if name is None:
            continue
        if len(vals) == 4:
            cx, cy, w, h = vals
        else:
            xs, ys = vals[0::2], vals[1::2]
            x0, x1, y0, y1 = min(xs), max(xs), min(ys), max(ys)
            cx, cy, w, h = (x0 + x1) / 2, (y0 + y1) / 2, x1 - x0, y1 - y0
        if h < min_h:
            continue
        ln = yolo_line(CID[name], cx, cy, w, h)
        if ln:
            out.append(ln)
    return out

def add_yolo_set(root, src, force=None, min_h=0.0, bg_ok=False, names=None):
    names = names if names is not None else read_names(root)
    n, counts = 0, Counter()
    for img, lbl in yolo_pairs(root):
        lines = remap_file(lbl, names, force, min_h)
        if lines:
            POOL.append((str(img), lines, src))
            n += 1
            for ln in lines:
                counts[CLASS_NAMES[int(ln.split()[0])]] += 1
        elif bg_ok:
            BG.append((str(img), src + '_bg'))
    SRC_STATS[src] = dict(counts)
    print(f'{src}: {n} images', dict(counts))
    return n

def fetch_many(jobs, threads=32):
    """jobs: list of (url, dest). Returns set of dest paths that exist."""
    ok = set()
    def one(job):
        url, dest = job
        dest = Path(dest)
        if dest.exists() and dest.stat().st_size > 1000:
            return dest
        try:
            urllib.request.urlretrieve(url, dest)
            if dest.stat().st_size > 1000:
                return dest
        except Exception:
            pass
        dest.unlink(missing_ok=True)
        return None
    with ThreadPoolExecutor(max_workers=threads) as ex:
        futs = [ex.submit(one, j) for j in jobs]
        for i, f in enumerate(as_completed(futs), 1):
            r = f.result()
            if r is not None:
                ok.add(str(r))
            if i % 500 == 0 or i == len(futs):
                print(f'  photos {i}/{len(futs)} ok {len(ok)}')
    return ok

def pool_summary():
    c = Counter()
    for _, lines, _ in POOL:
        for k in {int(l.split()[0]) for l in lines}:
            c[CLASS_NAMES[k]] += 1
    print('Images per class in pool:')
    for n in CLASS_NAMES:
        print(f'  {n:<11} {c.get(n, 0)}')
    print('  background', len(BG))

print('helpers ready')

## 3) Road damage: RDD2022 India + Japan

Real dashcam roads. `D40` becomes pothole, `D00`/`D10`/`D20` become crack. Road photos with no damage at all are kept as **background** so the model learns what a clean road looks like (fewer false alarms).

In [ ]:
#@title 3) RDD2022 India + Japan
RDD_BASE = ('https://bigdatacup.s3.ap-northeast-1.amazonaws.com/'
            '2022/CRDDC2022/RDD2022/Country_Specific_Data_CRDDC2022/')
RDD_MAP = {'D00': 'crack', 'D10': 'crack', 'D20': 'crack', 'D40': 'pothole'}

def rdd_add(root, src):
    imgs = {p.stem: p for p in Path(root).rglob('*') if p.suffix.lower() in IMG_EXT}
    n, nbg, counts = 0, 0, Counter()
    for x in Path(root).rglob('*.xml'):
        img = imgs.get(x.stem)
        if img is None:
            continue
        try:
            t = ET.parse(x).getroot()
        except ET.ParseError:
            continue
        W = float(t.findtext('size/width') or 0)
        H = float(t.findtext('size/height') or 0)
        if W <= 0 or H <= 0:
            with Image.open(img) as im:
                W, H = im.size
        lines, other = [], False
        for o in t.findall('object'):
            name = RDD_MAP.get((o.findtext('name') or '').strip().upper())
            if name is None:
                other = True
                continue
            b = o.find('bndbox')
            x0, y0 = float(b.findtext('xmin')), float(b.findtext('ymin'))
            x1, y1 = float(b.findtext('xmax')), float(b.findtext('ymax'))
            ln = yolo_line(CID[name], (x0 + x1) / 2 / W, (y0 + y1) / 2 / H, (x1 - x0) / W, (y1 - y0) / H)
            if ln:
                lines.append(ln)
                counts[name] += 1
        if lines:
            POOL.append((str(img), lines, src))
            n += 1
        elif not other:
            BG.append((str(img), src + '_bg'))
            nbg += 1
    SRC_STATS[src] = dict(counts)
    print(f'{src}: {n} images, {nbg} clean-road background', dict(counts))

for country in ('India', 'Japan'):
    try:
        z = DL / f'RDD2022_{country}.zip'
        saved = RAW_DRIVE / z.name
        if not z.exists() and saved.exists() and saved.stat().st_size > 50_000_000:
            print('copying', saved.name, 'from Drive')
            shutil.copy2(saved, z)
        if not z.exists():
            print('skip', country, '(old server is closed; cell 4 gets RDD India from Roboflow)')
            continue
        rdd_add(unzip(z, DL / f'rdd_{country.lower()}'), f'rdd_{country.lower()}')
    except Exception as e:
        print('skip', country, e)
pool_summary()

## 4) Roboflow sets

Paste your Roboflow API key. When a version is not given, the newest version is used. A project that fails is skipped.

- Speed bumps and potholes: the sets that worked in earlier passes.
- **Dashcam people:** BDD100K (10k dashcam frames). Only `person` is kept, `rider` is dropped. Frames with no people become road background.
- Donkey and buffalo: dedicated sets, because these two are rare everywhere.

In [ ]:
#@title 4) Roboflow downloads
ROBOFLOW_API_KEY = ''  #@param {type:"string"}
assert ROBOFLOW_API_KEY.strip(), 'Paste your Roboflow API key above'
from roboflow import Roboflow
rf = Roboflow(api_key=ROBOFLOW_API_KEY.strip())

def rf_get(ws, proj, folder, version=None):
    out = DL / folder
    if (out / '_OK').exists():
        return out
    p = rf.workspace(ws).project(proj)
    if version is None:
        nums = []
        try:
            for v in p.versions():
                try:
                    nums.append(int(str(getattr(v, 'version', '')).split('/')[-1]))
                except ValueError:
                    pass
        except Exception:
            pass
        tries = sorted(set(nums), reverse=True) or [3, 2, 1]
    else:
        tries = [version]
    err = None
    for v in tries:
        try:
            print(f'downloading {ws}/{proj} v{v}')
            p.version(v).download('yolov8', location=str(out), overwrite=True)
            (out / '_OK').write_text('ok')
            return out
        except Exception as e:
            err = e
    raise err

# (workspace, project, version or None, folder, options for add_yolo_set)
RF_JOBS = [
    ('speed-bump-detection', 'speed-bump-detection-se0eh', 15, 'rf_bump_v15', {}),
    ('alia-khalifa', 'detecting-speed-bumps', 4, 'rf_bump_india', {}),
    ('pothole-detection-1nczj', 'speed-unmarked-bumb', 1, 'rf_bump_unmarked', {}),
    ('nsip-project', 'road-degradation-beta', 1, 'rf_road_degradation', {}),
    ('brad-dwyer', 'pothole-voxrl', 1, 'rf_pothole_brad', {}),
    ('project-jnlc6', 'pothole-detection-yolov8', 1, 'rf_pothole_v8', {}),
    ('sjtu-xsfq7', 'rdd2022_india', 5, 'rf_rdd_india', {'bg_ok': True}),
    ('sids-garage', 'donkey-lr7xx', None, 'rf_donkey', {}),
    ('ptr-hergu', 'buffalo-detection-ehb4t', None, 'rf_buffalo_ptr', {}),
    ('genius-lab', 'buffallo-april-21', None, 'rf_buffalo_genius', {}),
    ('nirma-university-cmc5a', 'buffalo-mgttx', None, 'rf_buffalo_nirma', {}),
    ('animal-detection-using-yolov8', 'animal-detection-using-yolov8', None, 'rf_animals', {}),
]
for ws, proj, ver, folder, opts in RF_JOBS:
    try:
        add_yolo_set(rf_get(ws, proj, folder, ver), folder, **opts)
    except Exception as e:
        print('skip', folder, e)

# Dashcam people. Keep boxes at least 2% of frame height so far pedestrians stay in.
bdd_done = False
for ws, proj, ver in (('bdd100k-zljaf', 'bdd100k_0808', None),
                      ('pedro-azevedo-3c9ol', 'bdd100k-3zgda', 4)):
    if bdd_done:
        break
    try:
        root = rf_get(ws, proj, 'rf_bdd_' + proj.replace('-', '_'), ver)
        bdd_done = add_yolo_set(root, 'bdd', min_h=0.02, bg_ok=True) > 0
    except Exception as e:
        print('skip BDD', proj, e)
print('BDD people added:', bdd_done)
pool_summary()

In [ ]:
#@title 5) Pothole zip + African wildlife (buffalo)
try:
    z = download('https://www.dropbox.com/s/qvglw8pqo16769f/pothole_dataset_v8.zip?dl=1',
                 DL / 'pothole_dataset_v8.zip')
    add_yolo_set(unzip(z, DL / 'pothole_v8'), 'pothole_v8', force='pothole')
except Exception as e:
    print('skip pothole zip', e)

try:
    z = download('https://github.com/ultralytics/assets/releases/download/v0.0.0/african-wildlife.zip',
                 DL / 'african-wildlife.zip', 20_000_000)
    root = unzip(z, DL / 'african_wildlife')
    # Class 0 is buffalo. Elephant, rhino and zebra are dropped.
    add_yolo_set(root, 'african_buffalo', names={0: 'buffalo'})
except Exception as e:
    print('skip african wildlife', e)
pool_summary()

## 6) COCO: street people + dog, cat, horse, cow

Downloads only the label file (240 MB), then fetches just the photos we need. **People are only taken from street photos** (the photo must also contain a car, bus, truck, motorbike, bicycle or traffic light). Crowd photos are skipped because their people are not boxed one by one.

In [ ]:
#@title 6) COCO subset
COCO_CAP = {'person': 2600, 'dog': 1800, 'cat': 1500, 'horse': 1800, 'cow': 1800}
COCO_OURS = {1: 'person', 17: 'cat', 18: 'dog', 19: 'horse', 21: 'cow'}
ROAD_CONTEXT = {2, 3, 4, 6, 8, 10}

try:
    z = download('http://images.cocodataset.org/annotations/annotations_trainval2017.zip',
                 DL / 'coco_ann.zip', 200_000_000)
    jpath = DL / 'annotations' / 'instances_train2017.json'
    if not jpath.exists():
        with zipfile.ZipFile(z) as zz:
            zz.extract('annotations/instances_train2017.json', DL)
    data = json.loads(jpath.read_text())
    images = {im['id']: im for im in data['images']}
    lines_by, ctx, crowd = defaultdict(list), set(), set()
    for a in data['annotations']:
        c, iid = a['category_id'], a['image_id']
        if c in ROAD_CONTEXT:
            ctx.add(iid)
        if c not in COCO_OURS:
            continue
        if a.get('iscrowd'):
            crowd.add(iid)
            continue
        im = images[iid]
        x, y, w, h = a['bbox']
        if w < 4 or h < 4:
            continue
        W, H = im['width'], im['height']
        ln = yolo_line(CID[COCO_OURS[c]], (x + w / 2) / W, (y + h / 2) / H, w / W, h / H)
        if ln:
            lines_by[iid].append(ln)
    del data

    rng = random.Random(7)
    chosen = set()
    for cname, cap in COCO_CAP.items():
        k = CID[cname]
        ids = [i for i, ls in lines_by.items()
               if i not in crowd and any(int(l.split()[0]) == k for l in ls)
               and (cname != 'person' or i in ctx)]
        rng.shuffle(ids)
        chosen.update(ids[:cap])
        print(cname, 'candidates', len(ids), 'taking', min(cap, len(ids)))

    coco_dir = DL / 'coco'
    coco_dir.mkdir(exist_ok=True)
    jobs = [(f"http://images.cocodataset.org/train2017/{images[i]['file_name']}",
             coco_dir / images[i]['file_name']) for i in chosen]
    ok = fetch_many(jobs)
    counts = Counter()
    for i in chosen:
        p = str(coco_dir / images[i]['file_name'])
        if p in ok:
            POOL.append((p, lines_by[i], 'coco'))
            for l in lines_by[i]:
                counts[CLASS_NAMES[int(l.split()[0])]] += 1
    SRC_STATS['coco'] = dict(counts)
    print('coco boxes', dict(counts))
except Exception as e:
    print('skip COCO', e)
pool_summary()

## 7) Open Images: goat, donkey (mule), horse, cow

Uses the Open Images box list (2.2 GB, reused from Drive if Pass 3 saved it), then fetches only the photos needed. Drawings and group boxes are skipped.

In [ ]:
#@title 7) Open Images subset
OI_CAP = {'goat': 1800, 'donkey': 1500, 'horse': 900, 'cow': 900}
OI_FALLBACK_MID = {'Goat': '/m/03fwl', 'Mule': '/m/0dbzx', 'Horse': '/m/03k3r',
                   'Cattle': '/m/01xq0k1', 'Bull': '/m/0cnyhnx'}
OI_TO_OURS = {'Goat': 'goat', 'Mule': 'donkey', 'Donkey': 'donkey', 'Horse': 'horse',
              'Cattle': 'cow', 'Bull': 'cow'}

try:
    mids = {}
    try:
        desc = download('https://storage.googleapis.com/openimages/v5/class-descriptions-boxable.csv',
                        DL / 'oi_classes.csv', 5_000)
        for row in csv.reader(desc.open(encoding='utf-8', errors='ignore')):
            if len(row) >= 2 and row[1] in OI_TO_OURS:
                mids[row[0]] = OI_TO_OURS[row[1]]
    except Exception as e:
        print('class list failed, using fallback ids', e)
    if not mids:
        mids = {m: OI_TO_OURS[n] for n, m in OI_FALLBACK_MID.items()}
    print('Open Images classes', mids)

    bbox_csv = DL / 'oidv6-train-annotations-bbox.csv'
    drive_csv = RAW_DRIVE / 'oidv6-train-annotations-bbox.csv'
    if not bbox_csv.exists() and drive_csv.exists() and drive_csv.stat().st_size > 1e9:
        print('copying box list from Drive')
        shutil.copy2(drive_csv, bbox_csv)
    download('https://storage.googleapis.com/openimages/v6/oidv6-train-annotations-bbox.csv',
             bbox_csv, 1_000_000_000)

    chosen = {n: set() for n in OI_CAP}
    boxes = defaultdict(list)
    with bbox_csv.open(newline='', encoding='utf-8', errors='ignore') as f:
        reader = csv.reader(f)
        head = next(reader)
        ix = {k: head.index(k) for k in ('ImageID', 'LabelName', 'Confidence', 'XMin', 'XMax',
                                         'YMin', 'YMax', 'IsGroupOf', 'IsDepiction')}
        for row in reader:
            name = mids.get(row[ix['LabelName']])
            if name is None or row[ix['Confidence']] not in ('1', '1.0'):
                continue
            if row[ix['IsGroupOf']] == '1' or row[ix['IsDepiction']] == '1':
                continue
            iid = row[ix['ImageID']]
            if iid not in chosen[name] and len(chosen[name]) >= OI_CAP[name]:
                continue
            x0, x1 = float(row[ix['XMin']]), float(row[ix['XMax']])
            y0, y1 = float(row[ix['YMin']]), float(row[ix['YMax']])
            if x1 - x0 < 0.02 or y1 - y0 < 0.02:
                continue
            chosen[name].add(iid)
            boxes[iid].append(yolo_line(CID[name], (x0 + x1) / 2, (y0 + y1) / 2, x1 - x0, y1 - y0))
    print('Open Images photos chosen', {n: len(s) for n, s in chosen.items()})

    oi_dir = DL / 'oi'
    oi_dir.mkdir(exist_ok=True)
    jobs = [(f'https://open-images-dataset.s3.amazonaws.com/train/{iid}.jpg', oi_dir / f'{iid}.jpg')
            for iid in boxes]
    ok = fetch_many(jobs)
    counts = Counter()
    for iid, lines in boxes.items():
        p = str(oi_dir / f'{iid}.jpg')
        lines = [l for l in lines if l]
        if p in ok and lines:
            POOL.append((p, lines, 'openimages'))
            for l in lines:
                counts[CLASS_NAMES[int(l.split()[0])]] += 1
    SRC_STATS['openimages'] = dict(counts)
    print('openimages boxes', dict(counts))
except Exception as e:
    print('skip Open Images', e)
pool_summary()

## 8) Balance and write the dataset

- Each photo counts toward its **rarest** class, so a photo with a donkey and a person helps donkey.
- Inside a class, sources take turns (round robin), so people are half dashcam (BDD) and half street (COCO) instead of all one kind.
- Test set: at least 150 photos per class (or 10%, whichever is bigger). Same split is used to score old vs new.
- Photos are resized to 800 px on the long side. Training reads them faster and the boxes do not change.

**Check the printed table.** A class under 300 images will stay weak.

In [ ]:
#@title 8) Balance + write
TARGET = {
    'pothole': 2600, 'crack': 2000, 'speed_bump': 2200, 'person': 2800,
    'cow': 1800, 'buffalo': 1500, 'dog': 1600, 'cat': 1300,
    'horse': 1600, 'donkey': 1500, 'goat': 1600,
}
BG_TARGET = 1600
VAL_FRAC, VAL_MIN = 0.10, 150
MAX_SIDE = 800

rng = random.Random(42)
seen, pool = set(), []
for item in POOL:
    if item[0] not in seen:
        seen.add(item[0])
        pool.append(item)

avail = Counter()
for _, lines, _ in pool:
    for k in {int(l.split()[0]) for l in lines}:
        avail[k] += 1

by_class = defaultdict(lambda: defaultdict(list))
for item in pool:
    ids = {int(l.split()[0]) for l in item[1]}
    primary = min(ids, key=lambda k: avail[k])
    by_class[primary][item[2]].append(item)

def round_robin(groups, n):
    lists = [g[:] for g in groups.values()]
    for g in lists:
        rng.shuffle(g)
    out = []
    while len(out) < n and any(lists):
        for g in lists:
            if g and len(out) < n:
                out.append(g.pop())
    return out

split_items = {'train': [], 'val': []}
plan = []
for name in CLASS_NAMES:
    items = round_robin(by_class[CID[name]], TARGET[name])
    n_val = min(max(VAL_MIN, int(len(items) * VAL_FRAC)), int(len(items) * 0.3))
    split_items['val'] += items[:n_val]
    split_items['train'] += items[n_val:]
    plan.append((name, avail[CID[name]], len(items), n_val))

bg_groups = defaultdict(list)
for p, src in dict.fromkeys(BG):
    bg_groups[src].append((p, [], src))
bg_items = round_robin(bg_groups, BG_TARGET)
n_bg_val = int(len(bg_items) * VAL_FRAC)
split_items['val'] += bg_items[:n_bg_val]
split_items['train'] += bg_items[n_bg_val:]

if MIX.exists():
    shutil.rmtree(MIX)
for s in ('train', 'val'):
    (MIX / 'images' / s).mkdir(parents=True)
    (MIX / 'labels' / s).mkdir(parents=True)

def write_one(args):
    idx, split, (src_path, lines, src) = args
    stem = f'{src}_{idx:06d}'
    try:
        with Image.open(src_path) as im:
            im = im.convert('RGB')
            w, h = im.size
            scale = MAX_SIDE / max(w, h)
            if scale < 1:
                im = im.resize((max(1, round(w * scale)), max(1, round(h * scale))), Image.BILINEAR)
            im.save(MIX / 'images' / split / f'{stem}.jpg', quality=90)
    except Exception:
        return None
    (MIX / 'labels' / split / f'{stem}.txt').write_text('\n'.join(lines) + ('\n' if lines else ''))
    return split, lines

jobs = []
for split, items in split_items.items():
    for i, it in enumerate(items):
        jobs.append((len(jobs), split, it))

files = Counter()
img_count = {s: Counter() for s in ('train', 'val')}
box_count = {s: Counter() for s in ('train', 'val')}
with ThreadPoolExecutor(max_workers=8) as ex:
    for i, r in enumerate(ex.map(write_one, jobs), 1):
        if r is None:
            continue
        split, lines = r
        files[split] += 1
        if not lines:
            img_count[split]['background'] += 1
        for k in {int(l.split()[0]) for l in lines}:
            img_count[split][CLASS_NAMES[k]] += 1
        for l in lines:
            box_count[split][CLASS_NAMES[int(l.split()[0])]] += 1
        if i % 2000 == 0:
            print(f'  wrote {i}/{len(jobs)}')

(MIX / 'data.yaml').write_text(
    f'path: {MIX}\ntrain: images/train\nval: images/val\nnames:\n'
    + ''.join(f'  {i}: {n}\n' for i, n in enumerate(CLASS_NAMES))
)

print()
print(f"{'class':<11} {'available':>9} {'picked':>7} {'train img':>9} {'val img':>8} {'train box':>9}")
for name, av, picked, nv in plan:
    flag = '   <-- LOW' if img_count['train'][name] < 300 else ''
    print(f"{name:<11} {av:>9} {picked:>7} {img_count['train'][name]:>9} {img_count['val'][name]:>8} {box_count['train'][name]:>9}{flag}")
print(f"{'background':<11} {len(BG):>9} {len(bg_items):>7} {img_count['train']['background']:>9} {img_count['val']['background']:>8}")
print('files', dict(files))
(RESULTS / 'dataset_counts.json').write_text(json.dumps({
    'plan': plan, 'images': {s: dict(c) for s, c in img_count.items()},
    'boxes': {s: dict(c) for s, c in box_count.items()}, 'sources': SRC_STATS,
}, indent=1))
assert files['train'] > 5000, 'Dataset too small. Scroll up for skip messages.'

In [ ]:
#@title 9) Save dataset zip to Drive (lets you resume after a disconnect)
local_zip = Path('/content/pass5_dataset.zip')
subprocess.check_call(f'cd /content && zip -0 -qr {local_zip} {MIX.name}', shell=True)
shutil.copy2(local_zip, DATASET_ZIP)
local_zip.unlink()
print('saved', DATASET_ZIP, f'{DATASET_ZIP.stat().st_size/1e9:.2f} GB')

## Restore (only after a disconnect)

Skip this on the first run. After a disconnect: run cell 1, then this, then cell 12 (Resume).

In [ ]:
#@title Restore dataset from Drive
if (MIX / 'data.yaml').exists():
    print('dataset already on this machine')
else:
    assert DATASET_ZIP.exists(), 'No dataset zip on Drive. Run cells 2-9 again.'
    shutil.copy2(DATASET_ZIP, '/content/pass5_dataset.zip')
    subprocess.check_call('cd /content && unzip -q pass5_dataset.zip && rm pass5_dataset.zip', shell=True)
    print('restored', MIX, (MIX / 'data.yaml').exists())

In [ ]:
#@title 10) Old model score on the new test set
import json
from ultralytics import YOLO

assert torch.cuda.is_available(), 'Runtime > Change runtime type > T4 GPU, then run cell 1, cell 2 and Restore.'
assert (MIX / 'data.yaml').exists(), 'Run the Restore cell first'

def per_class(model_path, tag):
    m = YOLO(str(model_path)).val(data=str(MIX / 'data.yaml'), imgsz=640, batch=16,
                                  device=0, plots=False, verbose=False)
    out = {'all': {'map50': float(m.box.map50), 'map': float(m.box.map)}}
    for j, k in enumerate(m.box.ap_class_index):
        out[CLASS_NAMES[int(k)]] = {
            'map50': float(m.box.ap50[j]),
            'p': float(m.box.p[j]),
            'r': float(m.box.r[j]),
        }
    (RESULTS / f'{tag}.json').write_text(json.dumps(out, indent=1))
    return out

old = per_class(OLD_BEST, 'old_model')
print('old model mAP50', round(old['all']['map50'], 3))
for n in CLASS_NAMES:
    if n in old:
        print(f"  {n:<11} mAP50 {old[n]['map50']:.3f}  recall {old[n]['r']:.3f}")

## 11) Train (overnight)

Starts from your current best model, so it keeps what it already knows. `last.pt` is saved to Drive after every epoch. Keep the tab open. If you get `CUDA out of memory`, change `batch=16` to `batch=8` and rerun only this cell.

In [ ]:
#@title 11) Train
import gc
from ultralytics import YOLO

assert torch.cuda.is_available(), 'Runtime > Change runtime type > T4 GPU, then run cell 1, cell 2 and Restore.'
assert (MIX / 'data.yaml').exists(), 'Run cell 8 (or Restore) first in this runtime'
gc.collect()
torch.cuda.empty_cache()
print('Starting from', OLD_BEST)

model = YOLO(str(OLD_BEST))
model.train(
    data=str(MIX / 'data.yaml'),
    epochs=40,
    patience=12,
    imgsz=640,
    batch=16,
    workers=2,
    device=0,
    project=str(RUN),
    name='finetune',
    exist_ok=True,
    pretrained=True,
    optimizer='SGD',
    lr0=0.004,
    lrf=0.05,
    cos_lr=True,
    warmup_epochs=2,
    cache=False,
    amp=True,
    plots=True,
    mosaic=1.0,
    close_mosaic=8,
    mixup=0.05,
    scale=0.5,
    translate=0.1,
    degrees=0.0,
    fliplr=0.5,
    hsv_h=0.015,
    hsv_s=0.6,
    hsv_v=0.5,
)
print('New best', RUN / 'finetune/weights/best.pt')

In [ ]:
#@title 12) Resume (only if Train stopped halfway)
from ultralytics import YOLO
last = RUN / 'finetune/weights/last.pt'
print('last.pt', last.exists(), last)
assert (MIX / 'data.yaml').exists(), 'Run the Restore cell first'
assert last.exists(), 'No last.pt yet. Run cell 11 instead.'
YOLO(str(last)).train(resume=True)

In [ ]:
#@title 13) Old vs new, per class
NEW_BEST = RUN / 'finetune/weights/best.pt'
assert NEW_BEST.exists(), NEW_BEST
old_path = RESULTS / 'old_model.json'
old = json.loads(old_path.read_text()) if old_path.exists() else per_class(OLD_BEST, 'old_model')
new = per_class(NEW_BEST, 'new_model')

print(f"{'class':<11} {'old mAP50':>9} {'new mAP50':>9} {'old recall':>10} {'new recall':>10}")
wins = 0
for n in CLASS_NAMES:
    o, w = old.get(n), new.get(n)
    if not o or not w:
        print(f'{n:<11} missing from test set')
        continue
    better = w['map50'] >= o['map50']
    wins += better
    print(f"{n:<11} {o['map50']:>9.3f} {w['map50']:>9.3f} {o['r']:>10.3f} {w['r']:>10.3f}  {'better' if better else 'WORSE'}")
print(f"{'all':<11} {old['all']['map50']:>9.3f} {new['all']['map50']:>9.3f}")
print(f'new model wins {wins} of {len(CLASS_NAMES)} classes')

In [ ]:
#@title 14) Export phone file
from ultralytics import YOLO
NEW_BEST = RUN / 'finetune/weights/best.pt'
out = YOLO(str(NEW_BEST)).export(format='tflite', imgsz=640, int8=False)
out = Path(out)
if out.suffix != '.tflite':
    found = sorted(out.rglob('*float32*.tflite')) if out.is_dir() else []
    found = found or sorted(out.parent.rglob('*float32*.tflite')) or sorted(out.parent.rglob('*.tflite'))
    assert found, f'no .tflite found near {out}'
    out = found[0]
dest = EXPORTS / 'raasta_m2_m5_pass5.tflite'
shutil.copy2(out, dest)
shutil.copy2(NEW_BEST, EXPORTS / 'raasta_m2_m5_pass5.pt')

import tensorflow as tf
it = tf.lite.Interpreter(model_path=str(dest))
it.allocate_tensors()
print('input', it.get_input_details()[0]['shape'], it.get_input_details()[0]['dtype'])
print('output', it.get_output_details()[0]['shape'])
print('Download', dest)
print('App expects input [1 640 640 3] float32 and output [1 15 8400].')